#### Import libraries

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as SQL_FUNCTIONS

#### Create databricks widgets for easier alternations and testing

Improve readability and make it quicker to test out other values used in databricks tool

In [0]:

dbutils.widgets.text("bronze_table", "workspace.bda_taxi.taxi_bronze")
dbutils.widgets.text("silver_table", "workspace.bda_taxi.taxi_silver")


# Already created table manually by uploading the csv file to Catalog, pulling from there.
dbutils.widgets.text("zone_lookup_table", "workspace.bda_taxi.taxi_zone_lookup")
dbutils.widgets.dropdown("use_zone_lookup", "true", ["true", "false"])

# Outlier trimming quantiles
## TODO: Setting to 0.00 and 1 temporarily as need to do evaluations in next steps to decide if i should trim or not.
dbutils.widgets.text("outlier_low_q", "0.00")
dbutils.widgets.text("outlier_high_q", "1")

bronze_table = dbutils.widgets.get("bronze_table").strip()
silver_table = dbutils.widgets.get("silver_table").strip()

zone_lookup_table = dbutils.widgets.get("zone_lookup_table").strip()
use_zone_lookup = dbutils.widgets.get("use_zone_lookup").lower() == "true"

low_quantile = float(dbutils.widgets.get("outlier_low_q"))
high_quantile = float(dbutils.widgets.get("outlier_high_q"))

print("Bronze table:", bronze_table)
print("Silver table:", silver_table)
print("Use zone lookup:", use_zone_lookup)
print("Zone lookup table:", zone_lookup_table if zone_lookup_table else "(not set)")
print("Outlier quantiles:", low_quantile, high_quantile)

#### Create helper functions (cleaning, feature engineering, enrichment, writing)

In [0]:
def read_table(table_fqn: str) -> DataFrame:
    return spark.table(table_fqn)

def ensure_schema_exists_for_table(table_fqn: str) -> None:
    """
    Ensures the Unity Catalog schema exists.
    Expects table_fqn in the form: catalog.schema.table
    """
    parts = table_fqn.split(".")
    if len(parts) < 3:
        raise ValueError(f"Expected fully qualified table name (catalog.schema.table), got: {table_fqn}")
    schema_fqn = ".".join(parts[:2])
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_fqn}")

def apply_core_quality_rules(input_dataframe: DataFrame) -> DataFrame:
    """
    Core cleaning rules:
    - Required fields are non-null
    - Dropoff datetime is not earlier than pickup datetime
    - Numeric validity checks for key numeric fields
    - Fill common charge-related nulls with 0
    """
    required_columns = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID"]
    dataframe = input_dataframe
    for column_name in required_columns:
        dataframe = dataframe.filter(SQL_FUNCTIONS.col(column_name).isNotNull())

    dataframe = dataframe.filter(
        SQL_FUNCTIONS.col("tpep_dropoff_datetime") >= SQL_FUNCTIONS.col("tpep_pickup_datetime")
    )

    dataframe = dataframe.filter(
        (SQL_FUNCTIONS.col("trip_distance") > 0) &
        (SQL_FUNCTIONS.col("total_amount") > 0) &
        (SQL_FUNCTIONS.col("fare_amount") >= 0)
    )

    numeric_fill_values = {
        "passenger_count": 0.0,
        "extra": 0.0,
        "mta_tax": 0.0,
        "tip_amount": 0.0,
        "tolls_amount": 0.0,
        "improvement_surcharge": 0.0,
        "congestion_surcharge": 0.0,
        "airport_fee": 0.0
    }
    return dataframe.fillna(numeric_fill_values)

def trim_outliers_by_quantiles(
    input_dataframe: DataFrame,
    column_bounds: dict,
    relative_error: float = 0.01
) -> DataFrame:
    """
    Light outlier trimming using approximate quantiles.

    column_bounds example:
    {
      "trip_distance": (0.01, 0.99),
      "fare_amount": (0.01, 0.99)
    }
    """
    dataframe = input_dataframe
    for column_name, (low_q, high_q) in column_bounds.items():
        bounds = dataframe.approxQuantile(column_name, [low_q, high_q], relative_error)
        lower_bound, upper_bound = bounds[0], bounds[1]
        print(f"Outlier bounds for {column_name}: {lower_bound} .. {upper_bound}")
        dataframe = dataframe.filter(SQL_FUNCTIONS.col(column_name).between(lower_bound, upper_bound))
    return dataframe

def add_derived_fields(input_dataframe: DataFrame) -> DataFrame:
    """
    Adds:
    - pickup_date, pickup_hour, pickup_dow: day of week using Spark convention (1 = Sunday, ..., 7 = Saturday)
    - time_bucket (Night/Day/Evening - each is 8 hours, Night start at 00:00 to 07:59 etc.)
    - tip_recorded (was a tip possible to record i.e. payment_type is credit card)
    - tipped label (credit card only and amount greater than 0)
    """
    dataframe = (
        input_dataframe
        .withColumn("pickup_date", SQL_FUNCTIONS.to_date("tpep_pickup_datetime"))
        .withColumn("pickup_hour", SQL_FUNCTIONS.hour("tpep_pickup_datetime"))
        .withColumn("pickup_dow", SQL_FUNCTIONS.dayofweek("tpep_pickup_datetime"))  # 1..7
    )

    dataframe = dataframe.withColumn(
        "time_bucket",
        SQL_FUNCTIONS.when(SQL_FUNCTIONS.col("pickup_hour").between(0, 7), SQL_FUNCTIONS.lit("Night"))
        .when(SQL_FUNCTIONS.col("pickup_hour").between(8, 15), SQL_FUNCTIONS.lit("Day"))
        .otherwise(SQL_FUNCTIONS.lit("Evening"))
    )

    # If a tip is possible to observer on this row, if the user paid by credit card
    dataframe = dataframe.withColumn(
        "tip_recorded",
        SQL_FUNCTIONS.when(SQL_FUNCTIONS.col("payment_type") == 1, SQL_FUNCTIONS.lit(1))
                    .otherwise(SQL_FUNCTIONS.lit(0))
    )

    dataframe = dataframe.withColumn(
        "tipped",
        SQL_FUNCTIONS.when(
            (SQL_FUNCTIONS.col("payment_type") == 1) & (SQL_FUNCTIONS.col("tip_amount") > 0),
            SQL_FUNCTIONS.lit(1)
        ).otherwise(SQL_FUNCTIONS.lit(0))
    )

    return dataframe

def load_zone_lookup(zone_table_fqn: str) -> DataFrame:
    """
    Loads the taxi zone lookup:
    - a Unity Catalog table (preferred)
    Returns a standardised schema for joining.
    """
    zones = spark.table(zone_table_fqn)

    return zones.select(
        SQL_FUNCTIONS.col("LocationID").cast("int").alias("LocationID"),
        SQL_FUNCTIONS.col("Borough").alias("Borough"),
        SQL_FUNCTIONS.col("Zone").alias("Zone"),
        SQL_FUNCTIONS.col("service_zone").alias("service_zone")
    )

def enrich_with_zone_names(input_dataframe: DataFrame, zones_dataframe: DataFrame) -> DataFrame:
    """
    Adds:
    - PU_Borough, PU_Zone, PU_service_zone
    - DO_Borough, DO_Zone, DO_service_zone
    """
    pickup_zones = zones_dataframe.select(
        SQL_FUNCTIONS.col("LocationID").alias("PULocationID"),
        SQL_FUNCTIONS.col("Borough").alias("PU_Borough"),
        SQL_FUNCTIONS.col("Zone").alias("PU_Zone"),
        SQL_FUNCTIONS.col("service_zone").alias("PU_service_zone")
    )

    dropoff_zones = zones_dataframe.select(
        SQL_FUNCTIONS.col("LocationID").alias("DOLocationID"),
        SQL_FUNCTIONS.col("Borough").alias("DO_Borough"),
        SQL_FUNCTIONS.col("Zone").alias("DO_Zone"),
        SQL_FUNCTIONS.col("service_zone").alias("DO_service_zone")
    )

    return (
        input_dataframe
        .join(pickup_zones, on="PULocationID", how="left")
        .join(dropoff_zones, on="DOLocationID", how="left")
    )

def write_delta_table_overwrite(input_dataframe: DataFrame, target_table_fqn: str) -> None:
    """
    Overwrites a Delta table in Unity Catalog, updating schema if needed.
    """
    ensure_schema_exists_for_table(target_table_fqn)
    (
        input_dataframe
        .write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable(target_table_fqn)
    )

def run_sanity_checks(silver_dataframe: DataFrame, zones_enabled: bool) -> None:
    """
    Prints a small set of sanity checks that are useful for debugging and reporting.
    """
    silver_dataframe.selectExpr(
        "count(*) as rows",
        "min(tpep_pickup_datetime) as min_pickup",
        "max(tpep_pickup_datetime) as max_pickup",
        "min(trip_distance) as min_trip_distance",
        "max(trip_distance) as max_trip_distance",
        "min(fare_amount) as min_fare_amount",
        "max(fare_amount) as max_fare_amount"
    ).show(truncate=False)

    silver_dataframe.groupBy("payment_type").count().orderBy("payment_type").show(truncate=False)

    credit_card = silver_dataframe.filter(SQL_FUNCTIONS.col("payment_type") == 1)
    credit_card.selectExpr(
        "avg(tipped) as credit_card_tip_rate",
        "count(*) as credit_card_trips"
    ).show(truncate=False)

    silver_dataframe.groupBy("time_bucket").count().orderBy("time_bucket").show(truncate=False)

    if zones_enabled:
        silver_dataframe.select(
            SQL_FUNCTIONS.count(SQL_FUNCTIONS.when(SQL_FUNCTIONS.col("PU_Zone").isNull(), 1)).alias("null_PU_Zone"),
            SQL_FUNCTIONS.count(SQL_FUNCTIONS.when(SQL_FUNCTIONS.col("DO_Zone").isNull(), 1)).alias("null_DO_Zone")
        ).show()

#### Pipeline execution (Bronze to Silver)

In [0]:
bronze_dataframe = read_table(bronze_table)
print("Bronze rows:", bronze_dataframe.count())
display(bronze_dataframe.limit(5))

# Step 1: Core cleaning
cleaned_dataframe = apply_core_quality_rules(bronze_dataframe)
print("Rows after core cleaning:", cleaned_dataframe.count())

# Step 2: Outlier trimming (light)
outlier_columns = {
    "trip_distance": (low_quantile, high_quantile),
    "fare_amount": (low_quantile, high_quantile)
}
trimmed_dataframe = trim_outliers_by_quantiles(cleaned_dataframe, outlier_columns, relative_error=0.01)
print("Rows after outlier trimming:", trimmed_dataframe.count())

# Step 3: Derived fields
feature_enriched_dataframe = add_derived_fields(trimmed_dataframe)
display(
    feature_enriched_dataframe.select(
        "tpep_pickup_datetime", "pickup_hour", "time_bucket",
        "payment_type", "tip_amount", "tipped"
    ).limit(10)
)

# Step 4: Improve readability
if use_zone_lookup:
    zones_dataframe = load_zone_lookup(zone_lookup_table)
    print("Zone lookup rows:", zones_dataframe.count())
    display(zones_dataframe.limit(10))

    silver_dataframe = enrich_with_zone_names(feature_enriched_dataframe, zones_dataframe)
else:
    silver_dataframe = feature_enriched_dataframe

display(silver_dataframe.limit(10))

## Write Silver and validate

write_delta_table_overwrite(silver_dataframe, silver_table)

print("Saved Silver table:", silver_table)
saved_silver = read_table(silver_table)
print("Silver rows:", saved_silver.count())

run_sanity_checks(saved_silver, zones_enabled=use_zone_lookup)